# The Idea
Supply responsivess changes with reform. We have no data to model how this will happen, we can only rely on the literature and **exogenously** impose a value. Key sources are:

+ Hilber and Vermeulen (2016)
+ Drayton, Levell and Sturrock (2024)
+ Ball, Meen and Neygaard (2010)

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
from python.functions.bridge import run_bridge, run_scenario, backtest_one_step, build_bridge_inputs

In [ ]:
# Loading ARDL params
coefs = pd.read_csv("../../R/models/ardl_coefs_full.csv")
ecm = coefs[coefs["type"] == "ecm"].set_index("term")["estimate"]
lr = coefs[coefs["type"] == "longrun"].set_index("term")["estimate"]
    
# england master for init
master = pd.read_csv("../../data/python_master/england_master.csv")
master.columns.values[0] = "period"

master["lhstarts"] = np.log(master["starts"])
master["lrprc"]    = np.log(master["hprice"] / master["gdp_def"])
master["lrcc"]     = np.log(master["cc"] / master["gdp_def"])
master["lvol"]     = np.log(master["vol"])
master["r3"]       = master["rate"]

init = master[master["period"].isin(["2025Q1", "2025Q2", "2025Q3", "2025Q4"])][
    ["period", "lhstarts", "lrprc", "lvol", "r3", "lrcc"]
].reset_index(drop=True)

obr = pd.read_csv("../../data/python_master/OBR/obr_scenario.csv")
obr = obr[obr["period"] != "2025Q4"].reset_index(drop=True)  # drop overlap anchor row


# Diagnostic to see if ECM is constructed correctly
for q in ["2024Q4", "2025Q1", "2025Q2"]:
    backtest_one_step(master, q, ecm, lr)
    print()

In [ ]:
baseline = run_scenario(1.00, init, obr, ecm, lr)
uplift25 = run_scenario(1.25, init, obr, ecm, lr)
uplift50 = run_scenario(1.50, init, obr, ecm, lr)

comparison = pd.DataFrame({
    "baseline": baseline.set_index("period")["lhstarts"],
    "uplift_25": uplift25.set_index("period")["lhstarts"],
    "uplift_50": uplift50.set_index("period")["lhstarts"],
})

display(comparison)

In [ ]:
import os

B = build_bridge_inputs()
OUT_DIR = "../../data/outputs"
CF_DIR = f"{OUT_DIR}/elasticity_cf"
os.makedirs(CF_DIR, exist_ok=True)

for name, df in [("baseline", baseline), ("uplift_25", uplift25), ("uplift_50", uplift50)]:
    out = df[["period", "lhstarts"]].copy()
    out["period"] = out["period"].astype(str)
    out.to_csv(f"{CF_DIR}/{name}.csv", index=False)

cf = {name: run_bridge(f"{CF_DIR}/{name}.csv", "lhstarts",
                       B["seed"], B["p"], B["seasonal"], B["fy_map"], B["net_add"], B["actual_back"])
      for name in ["baseline", "uplift_25", "uplift_50"]}

target = 1_500_000
for name, d in cf.items():
    cumulative = sum(d.values())
    print(f"{name}: {cumulative:,.0f} ({100*cumulative/target:.1f}% of target, "
          f"shortfall {target-cumulative:,.0f})")

# Result
Raising long-run supply price elasticity leaves cumulative net additions essentially unchanged, difference only in the thousands. The reason is that the OBR demand path holds real house prices close to flat over the forecast horizon so the elasticity channel has no way to work. A more elastic supply curve does not do much if there is no movement along the curve.


The results agrees with the reform scenario: whether we have more units (OBR's ~170k) or higher supply price responsiveness the 1.5m target is missed.